# Figure S3

Empirical decay curves under uniform, symmetric, and asymmetric mutation. This notebook awaits twelve colleague-provided 75-generation raw files.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pickle
import subprocess
import sys
from pathlib import Path

import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec

from slide.utils import (
    FIGURE_LABEL_SIZE,
    FIGURE_LEGEND_SIZE,
    FIGURE_TICK_SIZE,
    FIGURE_TITLE_SIZE,
    PANEL_LETTER_SIZE,
    get_figures_dir,
    get_processed_data_dir,
    get_raw_data_dir,
    load_pickle,
    save_pickle,
)

OVERWRITE_RAW_PKL: bool = False
OVERWRITE_PROCESSED_PKL: bool = False
PLOT_ONLY: bool = True
SAVE_FIGURES: bool = True
PANEL_DPI: int = 350
SAVE_TYPES = ("pdf", "png", "eps")

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
REPO_ROOT = Path.cwd()
print(f"raw_data: {RAW_DATA_DIR}")
print(f"processed_data: {PROCESSED_DATA_DIR}")
print(f"figures: {FIGURES_DIR}")
print(f"PLOT_ONLY={PLOT_ONLY}, OVERWRITE_RAW_PKL={OVERWRITE_RAW_PKL}, OVERWRITE_PROCESSED_PKL={OVERWRITE_PROCESSED_PKL}")


def save_figure(fig: Figure, stem: str, *, bbox_inches: str = "tight") -> None:
    """Save a figure in PDF, PNG, and EPS formats.

    Parameters:
    - fig: Figure
        Matplotlib figure to save.
    - stem: str
        Filename stem without an extension.
    - bbox_inches: str
        Bounding-box mode passed to Matplotlib.

    Returns:
    - None
        Files are written below ``FIGURES_DIR``.
    """
    for suffix in SAVE_TYPES:
        destination = FIGURES_DIR / suffix
        destination.mkdir(parents=True, exist_ok=True)
        fig.savefig(destination / f"{stem}.{suffix}", dpi=PANEL_DPI, bbox_inches=bbox_inches)


def add_panel_letter(ax: Axes, letter: str) -> None:
    """Add a bold manuscript panel letter.

    Parameters:
    - ax: Axes
        Axis receiving the annotation.
    - letter: str
        Panel letter.

    Returns:
    - None
        The annotation is added directly to ``ax``.
    """
    ax.text(-0.14, 1.10, letter, transform=ax.transAxes, fontsize=PANEL_LETTER_SIZE,
            fontweight="bold", va="top", ha="left")

from slide.direvo_functions import get_single_decay_rate_IK_v2, model_function_IK_v2
from slide_config import get_slide_data_dir

LANDSCAPE_NAMES = ("GB1", "TrpB", "TEV", "ParD3")
LANDSCAPE_KEYS = ("gb1", "trpb", "tev", "pard3")
MUTATION_MODELS = ("nuc_uniform", "nuc_h_sapiens_sym", "nuc_e_coli")
SLIDE_DATA_DIR = Path(get_slide_data_dir())
PROCESSED_PATH = PROCESSED_DATA_DIR / "figureS3_empirical_decay_processed.pkl"


## Figure S3 Raw Products

In [ ]:
def figure_s3_required_paths(slide_data_dir: Path) -> list[Path]:
    """Return the twelve expected 75-step raw paths for Figure S3.

    Parameters:
    - slide_data_dir: Path
        Directory containing the colleague-provided all-start simulations.

    Returns:
    - list[Path]
        Required raw-data paths in panel order.
    """
    return [
        slide_data_dir / f"decay_curves_{landscape}_{model}_m0.1_all_starts_75steps.pkl"
        for model in MUTATION_MODELS
        for landscape in LANDSCAPE_KEYS
    ]


required_raw_paths = figure_s3_required_paths(SLIDE_DATA_DIR)
if OVERWRITE_RAW_PKL and not PLOT_ONLY:
    environment = os.environ.copy()
    environment.update({
        "SLIDE_NUM_STEPS": "75",
        "SLIDE_MUTATION_MODELS": ",".join(MUTATION_MODELS),
        "SLIDE_OVERWRITE": "1",
    })
    subprocess.run(
        [sys.executable, "scripts/empirical_landscape_decay_curves_codon_fast.py"],
        cwd=REPO_ROOT,
        env=environment,
        check=True,
    )
missing_raw_paths = [path for path in required_raw_paths if not path.exists()]
if missing_raw_paths:
    print("Missing Figure S3 raw products:")
    print("\n".join(f"  - {path}" for path in missing_raw_paths))


## Figure S3 Processing

In [ ]:
def process_figure_s3_payload(slide_data_dir: Path) -> dict[str, object]:
    """Process the twelve empirical decay products used by Figure S3.

    Parameters:
    - slide_data_dir: Path
        Directory containing 75-generation raw curves.

    Returns:
    - dict[str, object]
        Observed, fitted, and idealised curves for all twelve panels.

    Raises:
    - FileNotFoundError
        If any required colleague-provided raw product is absent.
    """
    required = figure_s3_required_paths(slide_data_dir)
    missing = [path for path in required if not path.exists()]
    if missing:
        listing = "\n".join(f"  - {path}" for path in missing)
        raise FileNotFoundError(f"Figure S3 requires the following 75-step raw files:\n{listing}")
    processed_dir = get_processed_data_dir()
    spectral = load_pickle(processed_dir / "spectral_rho_comparison.pkl")
    constants_path = processed_dir / "true_constants_nuc.pkl"
    constants = load_pickle(constants_path) if constants_path.exists() else {}
    panels = {}
    for model in MUTATION_MODELS:
        for landscape_key, landscape_name in zip(LANDSCAPE_KEYS, LANDSCAPE_NAMES, strict=True):
            path = slide_data_dir / f"decay_curves_{landscape_key}_{model}_m0.1_all_starts_75steps.pkl"
            with path.open("rb") as handle:
                raw = np.asarray(pickle.load(handle), dtype=float)
            start_curves = raw.mean(axis=2).reshape(-1, 75)
            observed = np.square(start_curves).mean(axis=0)
            scale = max(float(observed[0]), 1e-10)
            normalized = observed / scale
            fitted_rate, fitted_amplitude, fitted_constant = get_single_decay_rate_IK_v2(
                normalized, mut=0.1, num_steps=75
            )
            fitted = model_function_IK_v2(
                np.arange(75), fitted_rate, fitted_amplitude * scale,
                fitted_constant * scale, mut=0.1,
            )
            constant_key = (landscape_name, model)
            true_amplitude, true_constant = constants.get(constant_key, (1.0, 0.0))
            spectral_key = model if model != "nuc_e_coli" else "nuc_e_coli_sym"
            rho = spectral.get(landscape_name, {}).get(spectral_key, fitted_rate / 2)
            idealised = model_function_IK_v2(
                np.arange(75), rho * 2, true_amplitude * scale, true_constant * scale, mut=0.1
            )
            panels[(model, landscape_name)] = {
                "observed": observed, "fitted": fitted, "idealised": idealised,
                "true_constant": true_constant * scale, "fitted_constant": fitted_constant * scale,
            }
    return {
        "data": panels,
        "params": {"M": 75, "mutation_rate": 0.1, "models": MUTATION_MODELS},
        "metadata": {"paper_reference": "Figure S3"},
    }


if PROCESSED_PATH.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
    figure_s3_payload = load_pickle(PROCESSED_PATH)
elif PLOT_ONLY:
    if missing_raw_paths:
        listing = "\n".join(f"  - {path}" for path in missing_raw_paths)
        raise FileNotFoundError(f"Figure S3 awaits these colleague-provided files:\n{listing}")
    raise FileNotFoundError(f"PLOT_ONLY=True requires processed payload {PROCESSED_PATH}")
else:
    figure_s3_payload = process_figure_s3_payload(SLIDE_DATA_DIR)
    save_pickle(figure_s3_payload, PROCESSED_PATH)


## Individual Panels A–L

In [ ]:
def plot_s3_panel(ax: Axes, payload: dict[str, object], model: str, landscape: str) -> None:
    """Plot one empirical mutation-model decay panel.

    Parameters:
    - ax: Axes
        Axis receiving the panel.
    - payload: dict[str, object]
        Processed Figure S3 payload.
    - model: str
        Mutation-model key.
    - landscape: str
        Empirical-landscape display name.

    Returns:
    - None
        Panel artists are added directly to ``ax``.
    """
    panel = payload["data"][(model, landscape)]  # type: ignore[index]
    generations = np.arange(len(panel["observed"]))
    ax.plot(generations, panel["observed"], "k.", markersize=2.2, alpha=0.5, label="Data")
    ax.plot(generations, panel["idealised"], color="tab:blue", linewidth=1.3, label="Idealised")
    ax.plot(generations, panel["fitted"], color="tab:orange", linestyle="--", linewidth=1.3, label="Fitted")
    ax.axhline(panel["true_constant"], color="tab:blue", linestyle=":", linewidth=0.9, label=r"True $c$")
    ax.axhline(panel["fitted_constant"], color="tab:orange", linestyle="-.", linewidth=0.9, label=r"Fitted $c$")
    ax.set_title(landscape, fontsize=FIGURE_TITLE_SIZE)
    ax.set_xlabel(r"Generations $M$", fontsize=FIGURE_LABEL_SIZE)
    ax.set_ylabel(r"$G_\mu$", fontsize=FIGURE_LABEL_SIZE)
    ax.tick_params(labelsize=FIGURE_TICK_SIZE)
    ax.spines[["top", "right"]].set_visible(False)


panel_specs = [
    (letter, model, landscape)
    for letter, (model, landscape) in zip(
        "ABCDEFGHIJKL",
        [(model, landscape) for model in MUTATION_MODELS for landscape in LANDSCAPE_NAMES],
        strict=True,
    )
]
for letter, model, landscape in panel_specs:
    fig, ax = plt.subplots(figsize=(3.0, 2.5), dpi=PANEL_DPI)
    plot_s3_panel(ax, figure_s3_payload, model, landscape)
    ax.legend(fontsize=6, frameon=False)
    add_panel_letter(ax, letter)
    if SAVE_FIGURES:
        save_figure(fig, f"figure_S3{letter}")
    plt.show()


## Complete Figure S3

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(11, 8), dpi=PANEL_DPI, constrained_layout=True)
model_titles = ("Uniform mutation", "H. sapiens (symmetric)", "E. coli (asymmetric)")
for row, (model, row_title) in enumerate(zip(MUTATION_MODELS, model_titles, strict=True)):
    for column, landscape in enumerate(LANDSCAPE_NAMES):
        ax = axes[row, column]
        letter = "ABCDEFGHIJKL"[row * 4 + column]
        plot_s3_panel(ax, figure_s3_payload, model, landscape)
        add_panel_letter(ax, letter)
        if column == 0:
            ax.set_ylabel(row_title + "\n" + r"$G_\mu$", fontsize=8)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=5, fontsize=7, bbox_to_anchor=(0.5, -0.04), frameon=False)
if SAVE_FIGURES:
    save_figure(fig, "figure_S3")
plt.show()
